# In-class Problem



## Bayesian Optimisation walk-through: finding the best ageing temperature

The data in this notebook are illustrative The "experiments" come from a virtual lab: a made-up hardness curve plus random measurement noise. The shape is loosely inspired by precipitation hardening, but the numbers are not real measurements of any alloy. Using a virtual lab lets us reveal the true answer at the end and check how well BO did. 

## Step 1: The problem

An aluminium alloy gets harder when it is aged: held at a moderate temperature so that tiny precipitates form inside the grains.

Age at too low a temperature and the precipitates grow too slowly (under-aged, soft).
Age at too high a temperature and they grow too large (over-aged, soft again).
Somewhere in between is a peak.
Our task: for a fixed ageing time, find the ageing temperature between 100 °C and 260 °C that gives the highest Vickers hardness (HV).

Each experiment means one furnace run plus a hardness test, about a day of work. Our budget is 12 experiments: 4 to start, then 8 chosen by BO.

A grid of 1 °C steps would need 161 experiments. We cannot afford that, so we need to choose each experiment wisely.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm   #norm is a ready-made object representing the standard normal distribution N(0, 1).
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
#each mini-Gaussian has a variance, and each pair of mini-Gaussians has a covariance. 
#The kernel is the rule that fills in those numbers, and each of the three sklearn kernels answers one part of that question.
#Here we have three of them
#RBF: how quickly agreement fades with distance.
#ConstantKernel: how big the swings are overall.
#WhiteKernel: how much private, unshared randomness each measurement adds

import warnings
warnings.filterwarnings("ignore")        # hide optimiser convergence warnings in the book

In [ ]:
#These are all plotting parameters - just to set the stage for plots and then we don't have to update everytime
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E4E7EA", "grid.linewidth": 0.8,
                     "font.size": 10})
BLUE, BAND, ORANGE, INK, GREY = "#2F6DB5", "#BFD3EC", "#C8641E", "#1D2327", "#8A9199"


In [ ]:
# ---- The virtual lab (hidden relationship: this is the "unknown" truth) ----
def true_hardness(T):
    T = np.asarray(T, float)
    return (60 + 55 * np.exp(-0.5 * ((T - 190) / 28) ** 2)
               + 18 * np.exp(-0.5 * ((T - 135) / 14) ** 2))
# The equation is two Gaussians that vaguely represents an age-hardening procedure

lab_rng = np.random.default_rng(0)
def virtual_lab(T):
    # One hardness measurement: true value + noise (standard deviation 2 HV)
    return true_hardness(T) + lab_rng.normal(0, 2.0, np.shape(T))

In [ ]:
#Let's just see what this looks like
#But remember!!
#======= We cannot see this true relationship!========= It is an unknown in a real problem =========
#======= This is exactly what we need to dig out with Bayesian Optimization ==========


def plot_true_hardness(T_min=80, T_max=280, n_meas=15, seed=1):
    T = np.linspace(T_min, T_max, 400)                 # local: exists only inside this function
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(T, true_hardness(T), "k-", lw=2.5, label="true hardness")

    rng = np.random.default_rng(seed)                  # own RNG, so lab_rng isn't consumed
    T_meas = np.linspace(T_min + 10, T_max - 10, n_meas)
    y_meas = true_hardness(T_meas) + rng.normal(0, 2.0, n_meas)
    ax.plot(T_meas, y_meas, "o", color="C3", label="noisy measurements (σ = 2 HV)")

    ax.set_xlabel("Ageing temperature (°C)")
    ax.set_ylabel("Hardness (HV)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig, ax

plot_true_hardness()

## Step 2: The plan

Bayesian Optimisation repeats the same four steps until the budget runs out:

```text
   ┌──────────────────────────────────────────────────────────────┐
   │                                                              │
   ▼                                                              │
 A. FIT      a GP to every measurement so far                     │
 B. PREDICT  mean and uncertainty at every candidate temperature  │
 C. SCORE    each candidate with Expected Improvement             │
 D. MEASURE  at the highest-scoring temperature, add the result ──┘
```

Two small details in the code are worth pointing out before you read it:

- **Scaling:** we map 100–260 °C onto 0–1 before fitting. This keeps the length-scale bounds meaningful (the same idea as feature scaling earlier in the course).
- **Kernel:** `ConstantKernel × RBF + WhiteKernel`. These three parts learn the signal size, the length scale and the measurement noise. All three are fitted automatically inside `.fit()` by maximising the marginal likelihood.

In [ ]:
# ---- Starting point: four spread-out experiments ----
lab_rng = np.random.default_rng(0) 

T_measured = np.array([110.0, 150.0, 220.0, 250.0])        # ageing temperature, °C  - these are the starting measurements
H_measured = virtual_lab(T_measured)                       # hardness, HV - we obtain these values from running the real experiment
T_candidates = np.linspace(100, 260, 321)                  # temperatures we may try (0.5 °C steps)


In [ ]:
def scale(T):
    # Map 100–260 °C onto 0–1 (a column, as scikit-learn expects)
    return ((np.asarray(T) - 100) / 160).reshape(-1, 1)    # We are scaling the Temperatures
    #This is standard practice in GPR : it gives us a scaled value to 'tune' rather than relying on absolute values

In [ ]:
#If I run an experiment at this temperature, how much better than my current best result do I expect to get, on average?

def expected_improvement(mu, sd, best, xi=0.5):
    # How much better than `best` do we expect to do? (xi = small margin, in HV)
    improvement = mu - best - xi
    z = improvement / np.maximum(sd, 1e-9)
    return improvement * norm.cdf(z) + sd * norm.pdf(z)

#At each candidate temperature, the GP gives a mini-Gaussian for the hardness you’d measure there: mean (mu), sd (standard devaiation), best value so far, xi - a small margin
# Picture the bell curve at that temperature, shade the tail that lies above best + ξ, and EI is the average size of the improvement in that shaded tail.

In [ ]:
def fit_gp(T, H):
    kernel = (ConstantKernel(1.0, (1e-2, 1e2))
              * RBF(length_scale=0.2, length_scale_bounds=(0.02, 2.0))
              + WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-3, 1e-1)))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                  n_restarts_optimizer=5, random_state=0)
    return gp.fit(scale(T), H)
#The gp.fit() contains a few things like the kernel with optimized parameters (kernel_), The score the optimizer maximized (log_marginal_likelihood_value_)
# A copy of the training data, some pre-computed matrices

history = []
#Since our experiment  budget is 12, and we've already used up 4 for the initial measurements - we still have 8 left


for i in range(8):
    gp = fit_gp(T_measured, H_measured)                                  # A. fit
    mu, sd = gp.predict(scale(T_candidates), return_std=True)            # B. predict
    ei = expected_improvement(mu, sd, best=H_measured.max())             # C. score
    T_next = T_candidates[np.argmax(ei)]                                 #ei is an array with one EI value per candidate temperature, in the same order as T_candidates.
                                                                         # We find the location of the ei--> that corresponds to the next T
    history.append(dict(T=T_measured.copy(), H=H_measured.copy(),
                        mu=mu, sd=sd, ei=ei, T_next=T_next))
    H_next = virtual_lab(T_next)                                         # D. measure
    T_measured = np.append(T_measured, T_next)
    H_measured = np.append(H_measured, H_next)
    print(f"Experiment {i + 5:2d}: T = {T_next:5.1f} °C  ->  H = {H_next:5.1f} HV"
          f"   (best so far {H_measured.max():5.1f} HV)")

final_gp = fit_gp(T_measured, H_measured)
best = np.argmax(H_measured)
print(f"\nBest measured: {H_measured[best]:.1f} HV at {T_measured[best]:.1f} °C")
print("Learned kernel:", final_gp.kernel_)

## Step 3: Reading the first fit

Before BO chooses anything, look at what the GP believes after the four starting experiments.

- The **blue line** is the GP mean, its best guess of hardness.
- The **shaded band** is ±2 standard deviations, roughly a 95% range.
- The **band is thin** at the measured points and **wide** in the gaps between them.
- The lower panel is the **Expected Improvement** score. The orange line marks its peak: the next experiment.

In [ ]:
h = history[0]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5.2), sharex=True,
                               gridspec_kw={"height_ratios": [2.2, 1]})
ax1.fill_between(T_candidates, h["mu"] - 2 * h["sd"], h["mu"] + 2 * h["sd"],
                 color=BAND, lw=0, label="±2 std (uncertainty)")
ax1.plot(T_candidates, h["mu"], color=BLUE, lw=2, label="GP mean")
ax1.scatter(h["T"], h["H"], s=45, color=INK, zorder=3, label="measurements")
ax1.axvline(h["T_next"], color=ORANGE, lw=1.5, ls="--")
ax1.set_ylabel("Hardness (HV)")
ax1.set_title("After 4 experiments: what the GP believes", loc="left", fontsize=11)
ax1.legend(loc="upper left", frameon=False, fontsize=9)
ax2.fill_between(T_candidates, h["ei"], color=ORANGE, alpha=0.25, lw=0)
ax2.plot(T_candidates, h["ei"], color=ORANGE, lw=2)
ax2.axvline(h["T_next"], color=ORANGE, lw=1.5, ls="--")
ax2.annotate(f"next: {h['T_next']:.1f} °C", xy=(h["T_next"], h["ei"].max()),
             xytext=(8, -4), textcoords="offset points", color=INK, fontsize=9, va="top")
ax2.set_ylabel("Expected\nimprovement")
ax2.set_xlabel("Ageing temperature (°C)")
fig.tight_layout(); plt.show()

**What to notice.**

- **The GP looks strange.** Between the measurements the mean falls back to a flat average, and it makes narrow spikes at each point. With only four measurements, the GP cannot tell how smooth the curve is. Its fitted length scale hit the lower bound we allowed (0.02, about 3 °C), so each point only "informs" a tiny neighbourhood. This is the flat-likelihood problem: too little data to pin down the hyperparameters. Once 6 points are in, the fitted length scale jumps to about 0.32 (roughly 50 °C), and the fits look sensible from then on (Step 4).
- **EI still behaves sensibly.** Its highest peaks are just beside the best measurement (220 °C), so BO picks **217.5 °C** (exploitation). It also stays clearly above zero across the whole unexplored range, because the model cannot rule out something better there.

## Step 4: Watching the loop

Each panel shows the GP **before** one experiment. The orange dashed line marks where BO decided to measure next.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 5.4), sharex=True, sharey=True)
for k, (ax, h) in enumerate(zip(axes.flat, history)):
    ax.fill_between(T_candidates, h["mu"] - 2 * h["sd"], h["mu"] + 2 * h["sd"], color=BAND, lw=0)
    ax.plot(T_candidates, h["mu"], color=BLUE, lw=1.8)
    ax.scatter(h["T"], h["H"], s=22, color=INK, zorder=3)
    ax.axvline(h["T_next"], color=ORANGE, lw=1.5, ls="--")
    ax.set_title(f"Before experiment {k + 5} → {h['T_next']:.0f} °C", fontsize=9.5, loc="left")
    ax.set_ylim(35, 140)
for ax in axes[1]:
    ax.set_xlabel("Ageing T (°C)")
for ax in axes[:, 0]:
    ax.set_ylabel("Hardness (HV)")
fig.tight_layout(); plt.show()

**What to notice.**

- **Experiments 5–6:** BO checks next to the current best (217.5 °C), then jumps into the unexplored gap (169.5 °C). The measurement there is higher than anything seen before.
- **Experiments 7–8:** the band around 170–200 °C tightens, and BO finds a clearly higher value near 186 °C.
- **Experiments 9–12:** BO keeps sampling around 186–192 °C. The model is now confident the peak is here, so it spends the remaining budget refining it.

With a noise of about 2 HV, repeated measurements near the peak also help average out the noise. You could argue that the last two or three experiments would be better used elsewhere or saved. When to stop is a judgement call, discussed below.

## Step 5: What did the GP learn?

The printed kernel from the code cell above shows the fitted hyperparameters.

- **Length scale ≈ 0.24** in scaled units. Multiply by the 160 °C range to get about **38 °C**. Roughly: two temperatures closer than ~40 °C give similar hardness.
- **Noise level ≈ 0.013** in normalised units. `normalize_y=True` divides hardness by its standard deviation (about 18 HV for these twelve measurements), so this corresponds to a noise of about √0.013 × 18 ≈ **2.1 HV**. The virtual lab's true noise is 2 HV, so the GP's estimate is close.

Nobody typed in 38 °C or 2.1 HV. The GP learned both from twelve measurements by maximising the marginal likelihood.

## Step 6: Revealing the truth

In a real lab we would never see the true curve. Because this is a virtual lab, we can plot it and compare.

In [ ]:
mu, sd = final_gp.predict(scale(T_candidates), return_std=True)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.fill_between(T_candidates, mu - 2 * sd, mu + 2 * sd, color=BAND, lw=0, label="GP ±2 std (12 experiments)")
ax.plot(T_candidates, mu, color=BLUE, lw=2, label="GP mean")
ax.plot(T_candidates, true_hardness(T_candidates), color=GREY, lw=1.8, ls="--", label="true curve (hidden)")
ax.scatter(T_measured[:4], H_measured[:4], s=45, color=INK, zorder=3, label="starting experiments")
ax.scatter(T_measured[4:], H_measured[4:], s=45, facecolor="white", edgecolor=ORANGE, lw=2, zorder=3,
           label="chosen by BO")
ax.set_xlabel("Ageing temperature (°C)"); ax.set_ylabel("Hardness (HV)")
ax.legend(loc="upper left", frameon=False, fontsize=9)
fig.tight_layout(); plt.show()
T_star = T_candidates[np.argmax(true_hardness(T_candidates))]
print(f"True optimum: {T_star:.1f} °C, {true_hardness(T_star):.1f} HV")
print(f"BO's best measured point: {T_measured[best]:.1f} °C (true hardness there: {true_hardness(T_measured[best]):.1f} HV)")

**What to notice.**

- BO found the main peak within about 4 °C, using 12 experiments instead of 161.
- The GP is **accurate near the peak**, where it sampled heavily, and **less certain elsewhere**. The small secondary bump near 135 °C is poorly resolved, and that is fine: BO's job is to find the maximum, not to map the whole curve.
- The measured best (117.2 HV) is slightly above the true peak value (115.0 HV). That is measurement noise: a lucky reading. In a real report you would repeat the best condition to confirm it.

😊

The surrogate is our cheap imitation of the lab (the GP); 

the acquisition function (EI) reads the imitation to decide which real experiment is worth doing next.

## Step 7: Was it worth it? BO vs random sampling

One run could be luck. So we repeat the whole study **20 times**, each with different random starting points and noise, and compare BO with simply choosing the 8 extra temperatures at random.

For each run we track the **true** hardness at the best-measured temperature so far. That is the value you would actually get if you picked that condition.

In [ ]:
def best_true_curve(T, H):
    # True hardness at the best-MEASURED temperature after 1, 2, ... experiments
    return np.array([true_hardness(T[np.argmax(H[:n])]) for n in range(1, len(T) + 1)])

def run_bo(T_start, rng, n_iter=8):
    T = np.array(T_start, float); H = true_hardness(T) + rng.normal(0, 2.0, len(T))
    for _ in range(n_iter):
        g = fit_gp(T, H)
        m, s = g.predict(scale(T_candidates), return_std=True)
        Tn = T_candidates[np.argmax(expected_improvement(m, s, H.max()))]
        T = np.append(T, Tn); H = np.append(H, true_hardness(Tn) + rng.normal(0, 2.0))
    return T, H

bo_runs, rnd_runs = [], []
for s in range(20):
    r = np.random.default_rng(100 + s)
    T_start = np.sort(r.uniform(100, 260, 4))
    Tb, Hb = run_bo(T_start, np.random.default_rng(s))
    bo_runs.append(best_true_curve(Tb, Hb))
    Tr = np.concatenate([T_start, r.uniform(100, 260, 8)])
    Hr = true_hardness(Tr) + np.random.default_rng(s).normal(0, 2.0, len(Tr))
    rnd_runs.append(best_true_curve(Tr, Hr))
bo_runs, rnd_runs = np.array(bo_runs), np.array(rnd_runs)

n = np.arange(1, 13)
peak = true_hardness(T_candidates).max()
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.axhline(peak, color=GREY, lw=1, ls="--")
ax.text(1.1, peak + 0.8, f"true optimum {peak:.0f} HV", color=GREY, fontsize=9)
ax.axvspan(0.5, 4.5, color="#F1F3F5", lw=0)
ax.text(2.5, 72, "starting\nexperiments", ha="center", color=GREY, fontsize=9)
for runs, c, lab, mk in [(bo_runs, BLUE, "Bayesian optimisation", "o"), (rnd_runs, ORANGE, "random sampling", "s")]:
    ax.fill_between(n, np.percentile(runs, 25, axis=0), np.percentile(runs, 75, axis=0), color=c, alpha=0.15, lw=0)
    ax.plot(n, runs.mean(axis=0), color=c, lw=2, marker=mk, ms=6, label=lab + " (mean, IQR shaded)")
ax.set_xlim(0.5, 12.5); ax.set_ylim(70, 120); ax.set_xticks(n)
ax.set_xlabel("Number of experiments"); ax.set_ylabel("True hardness of best\ncondition found (HV)")
ax.legend(loc="lower right", frameon=False, fontsize=9)
fig.tight_layout(); plt.show()

within = peak - 2
print(f"Runs within 2 HV of the optimum after 12 experiments: "
      f"BO {np.sum(bo_runs[:, -1] >= within)}/20, random {np.sum(rnd_runs[:, -1] >= within)}/20")

**What to notice.**

- The first four experiments are identical for both methods, so the curves overlap there.
- At experiment 5, BO is briefly *behind* on average. Its first pick usually checks next to the current best, which rarely gives a big jump. That is not a failure: it is the model still learning the shape.
- From about experiment 7 onwards, BO pulls ahead and its spread (the shaded band) becomes narrower. After 12 experiments, BO is within 2 HV of the optimum in **19 of 20** runs, compared with **12 of 20** for random sampling.

Random sampling can get lucky, but it cannot use what it has learned. BO does, and the advantage shows as **reliability**: fewer bad outcomes for the same budget.

## Discussion: exploration vs exploitation

Every acquisition function balances two pulls:

- **Exploitation:** measure where the mean is high. This refines what you already know.
- **Exploration:** measure where the uncertainty is high. This checks you have not missed something.

Expected Improvement combines both automatically. A point scores well if its mean is high **or** its uncertainty is large enough that it *might* be high.

Another popular choice is the **Upper Confidence Bound (UCB)**: score = mean + κ × std. Here κ is a dial you set yourself: κ = 0 is pure exploitation, and a large κ favours exploration. Exercise 1 shows what goes wrong if you turn the dial all the way to exploitation.

:::{tip} Practical points for real experiments
- **Stopping:** stop when the budget is spent, or when the best EI value becomes small compared with the measurement noise.
- **Confirm the winner:** repeat the best condition a few times before reporting it.
- **Noise:** if you know your measurement repeatability, use it to set the lower bound of the noise level. With very few points, the GP cannot separate noise from signal on its own.
- **More than one input:** the same loop works with, say, temperature **and** time. Give each input its own length scale (`RBF(length_scale=[0.2, 0.2])`).
:::